In [1]:
# ==========================================
# CELL 1 — Imports & Config
# ==========================================
import pandas as pd
import numpy as np
import os
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Gold CSV output location (Files section, for manual inspection / Excel)
GOLD_FILES_PATH = "/lakehouse/default/Files/gold"
os.makedirs(GOLD_FILES_PATH, exist_ok=True)


# ==========================================
# CELL 2 — Load Silver data
#           (replaces load_latest_silver_data() which scanned local parquet files)
# ==========================================
def load_silver_data() -> pd.DataFrame:
    logging.info("Loading data from Silver Lakehouse table...")
    df = spark.sql("SELECT * FROM silver_bursa_banks").toPandas()
    logging.info(f"Loaded {len(df)} rows from silver_bursa_banks.")
    return df


# ==========================================
# CELL 3 — Dimension tables (unchanged logic)
# ==========================================
def build_dim_stock(df: pd.DataFrame) -> pd.DataFrame:
    """One row per stock, with a surrogate key for star-schema joins."""
    dim_stock = df[['ticker_symbol', 'stock_name', 'exchange']].drop_duplicates().reset_index(drop=True)
    dim_stock['stock_id'] = dim_stock['ticker_symbol'].str.split('.').str[0].astype(int)
    return dim_stock[['stock_id', 'stock_name', 'ticker_symbol', 'exchange']]


def build_dim_date(df: pd.DataFrame) -> pd.DataFrame:
    """
    Full continuous calendar (not just trading days) so Power BI's time-intelligence
    functions (YTD, QTD, DATESBETWEEN, etc.) and date hierarchies work without gaps.
    """
    min_date = pd.to_datetime(df['date'].min())
    max_date = pd.to_datetime(df['date'].max())
    dim_date = pd.DataFrame({'date': pd.date_range(start=min_date, end=max_date, freq='D')})

    dim_date['date_id'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
    dim_date['year'] = dim_date['date'].dt.year
    dim_date['month'] = dim_date['date'].dt.month
    dim_date['month_name'] = dim_date['date'].dt.month_name()
    dim_date['quarter'] = dim_date['date'].dt.quarter
    dim_date['day_of_week'] = dim_date['date'].dt.day_name()
    dim_date['is_weekend'] = dim_date['date'].dt.dayofweek >= 5
    return dim_date


# ==========================================
# CELL 4 — RSI (unchanged logic)
# ==========================================
def calc_rsi(prices: pd.Series, window: int = 14) -> pd.Series:
    """
    RSI = 100 - (100 / (1 + RS)), RS = avg_gain / avg_loss.
    Days with insufficient history or zero avg_loss are left as NaN
    (blank in Power BI) rather than filled with a fake neutral 50.
    """
    delta = prices.diff()
    gain = delta.where(delta > 0, 0).rolling(window=window).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=window).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


# ==========================================
# CELL 5 — Fact table: daily metrics (unchanged logic)
# ==========================================
def build_fact_stock_prices(df: pd.DataFrame, dim_stock: pd.DataFrame) -> pd.DataFrame:
    fact = df.merge(dim_stock[['ticker_symbol', 'stock_id']], on='ticker_symbol', how='left')
    fact['date'] = pd.to_datetime(fact['date'])
    fact['date_id'] = fact['date'].dt.strftime('%Y%m%d').astype(int)
    fact = fact.sort_values(['stock_id', 'date'])

    fact['daily_return_pct'] = (fact.groupby('stock_id')['adj_close'].pct_change() * 100).round(2)
    fact['volatility_7'] = fact.groupby('stock_id')['daily_return_pct'].transform(
        lambda x: x.rolling(7).std()).round(2)
    fact['volatility_30'] = fact.groupby('stock_id')['daily_return_pct'].transform(
        lambda x: x.rolling(30, min_periods=1).std()).round(2)

    fact['sma_20'] = fact.groupby('stock_id')['close'].transform(
        lambda x: x.rolling(20, min_periods=1).mean()).round(2)
    fact['sma_50'] = fact.groupby('stock_id')['close'].transform(
        lambda x: x.rolling(50, min_periods=1).mean()).round(2)

    fact['rsi_14'] = fact.groupby('stock_id')['close'].transform(calc_rsi).round(2)

    fact['indexed_price'] = fact.groupby('stock_id')['adj_close'].transform(
        lambda x: (x / x.iloc[0]) * 100).round(2)

    fact['cumulative_max'] = fact.groupby('stock_id')['adj_close'].transform(lambda x: x.cummax())
    fact['drawdown_pct'] = ((fact['adj_close'] - fact['cumulative_max']) / fact['cumulative_max'] * 100).round(2)

    fact['daily_price_range'] = (fact['high'] - fact['low']).round(2)

    cols = ['stock_id', 'stock_name', 'date_id', 'open', 'high', 'low', 'close', 'adj_close', 'volume',
            'daily_return_pct', 'volatility_7', 'volatility_30',
            'sma_20', 'sma_50', 'rsi_14', 'indexed_price',
            'cumulative_max', 'drawdown_pct', 'daily_price_range']
    return fact[cols].reset_index(drop=True)


# ==========================================
# CELL 6 — Correlation matrix (unchanged logic)
# ==========================================
def build_correlation_matrix(fact: pd.DataFrame, dim_stock: pd.DataFrame) -> pd.DataFrame:
    pivot = fact.pivot(index='date_id', columns='stock_id', values='daily_return_pct')
    corr = pivot.corr()

    corr_long = corr.reset_index().melt(id_vars='stock_id', var_name='stock_id_b', value_name='correlation')
    corr_long = corr_long.rename(columns={'stock_id': 'stock_id_a'})
    corr_long['correlation'] = corr_long['correlation'].round(3)

    name_map = dim_stock.set_index('stock_id')['stock_name'].to_dict()
    corr_long['stock_name_a'] = corr_long['stock_id_a'].map(name_map)
    corr_long['stock_name_b'] = corr_long['stock_id_b'].map(name_map)

    return corr_long[['stock_id_a', 'stock_name_a', 'stock_id_b', 'stock_name_b', 'correlation']]


# ==========================================
# CELL 7 — Run the pipeline (build all 4 Gold tables)
# ==========================================
df = load_silver_data()

dim_stock = build_dim_stock(df)
dim_date = build_dim_date(df)
fact_prices = build_fact_stock_prices(df, dim_stock)
corr_matrix = build_correlation_matrix(fact_prices, dim_stock)

logging.info("=== Gold tables built in memory: dim_stock, dim_date, fact_stock_prices, fact_stock_correlation ===")
fact_prices.head()


# ==========================================
# CELL 8 — Save as managed Delta tables (for Power BI Direct Lake)
# ==========================================
gold_tables = {
    "dim_stock": dim_stock,
    "dim_date": dim_date,
    "fact_stock_prices": fact_prices,
    "fact_stock_correlation": corr_matrix,
}

for name, tbl in gold_tables.items():
    spark_df = spark.createDataFrame(tbl)
    spark_df.write.format("delta").mode("overwrite").saveAsTable(name)
    logging.info(f"✅ Saved Delta table -> {name}")


# ==========================================
# CELL 9 — ALSO save CSV copies into Lakehouse Files (for Excel / manual inspection)
# ==========================================
for name, tbl in gold_tables.items():
    csv_path = f"{GOLD_FILES_PATH}/{name}.csv"
    tbl.to_csv(csv_path, index=False)
    logging.info(f"✅ Saved CSV -> {csv_path}")

logging.info("=== Gold Processing Complete. Ready for Power BI! ===")


# ==========================================
# CELL 10 — Quick verification
# ==========================================
spark.sql("SELECT * FROM fact_stock_prices LIMIT 10").show

StatementMeta(, c5b04349-5020-45d7-911b-6a3fbb6bedb7, 3, Finished, Available, Finished, False)

2026-09-04 03:39:34,805 - INFO - Loading data from Silver Lakehouse table...
2026-09-04 03:39:48,008 - INFO - Loaded 3295 rows from silver_bursa_banks.
2026-09-04 03:39:48,211 - INFO - === Gold tables built in memory: dim_stock, dim_date, fact_stock_prices, fact_stock_correlation ===
2026-09-04 03:39:55,757 - INFO - ✅ Saved Delta table -> dim_stock
2026-09-04 03:40:02,671 - INFO - ✅ Saved Delta table -> dim_date
2026-09-04 03:40:07,987 - INFO - ✅ Saved Delta table -> fact_stock_prices
2026-09-04 03:40:13,123 - INFO - ✅ Saved Delta table -> fact_stock_correlation
2026-09-04 03:40:13,234 - INFO - ✅ Saved CSV -> /lakehouse/default/Files/gold/dim_stock.csv
2026-09-04 03:40:13,306 - INFO - ✅ Saved CSV -> /lakehouse/default/Files/gold/dim_date.csv
2026-09-04 03:40:13,430 - INFO - ✅ Saved CSV -> /lakehouse/default/Files/gold/fact_stock_prices.csv
2026-09-04 03:40:13,520 - INFO - ✅ Saved CSV -> /lakehouse/default/Files/gold/fact_stock_correlation.csv
2026-09-04 03:40:13,520 - INFO - === Gold P

<bound method DataFrame.show of DataFrame[stock_id: bigint, stock_name: string, date_id: bigint, open: double, high: double, low: double, close: double, adj_close: double, volume: bigint, daily_return_pct: double, volatility_7: double, volatility_30: double, sma_20: double, sma_50: double, rsi_14: double, indexed_price: double, cumulative_max: double, drawdown_pct: double, daily_price_range: double]>